In [10]:
%pip install boto3==1.34.0 pandas

Note: you may need to restart the kernel to use updated packages.


In [11]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="anfa-admin",         
    aws_secret_access_key="anfa-password-2026",
    region_name="us-east-1",
)

# On s'assure que le bucket anfa-raw existe bien
try:
    s3.create_bucket(Bucket="anfa-raw")
    print("Bucket 'anfa-raw' initialisé !")
except Exception:
    print("Le bucket existe déjà.")

import os
chemin_local = "/home/jovyan/work/../data/referentiel/lignes.csv"
if os.path.exists(chemin_local):
    s3.upload_file(chemin_local, "anfa-raw", "referentiel/lignes.csv")
    print("Fichier 'lignes.csv' téléversé avec succès dans MinIO !")

s3.list_buckets()

Le bucket existe déjà.


{'ResponseMetadata': {'RequestId': '18BBC6E237E78A4C',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'content-length': '366',
   'content-type': 'application/xml',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '18BBC6E237E78A4C',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '843',
   'x-ratelimit-remaining': '843',
   'x-xss-protection': '1; mode=block',
   'date': 'Tue, 23 Jun 2026 17:45:52 GMT'},
  'RetryAttempts': 0},
 'Buckets': [{'Name': 'anfa-raw',
   'CreationDate': datetime.datetime(2026, 6, 23, 17, 42, 45, 263000, tzinfo=tzlocal())}],
 'Owner': {'DisplayName': 'minio',
  'ID': '02d6176db174dc93cb1b899f7c6078f08654445fe8cf1b6ce98d8855f66bdbf4'}}

In [12]:
reponse = s3.list_objects_v2(Bucket="anfa-raw", Prefix="referentiel/")
for obj in reponse.get("Contents", []):
    print(f"{obj['Key']} ({obj['Size']} octets)")

referentiel/test.txt (21 octets)


In [14]:
s3.upload_file("lignes.csv", "anfa-raw", "referentiel/lignes.csv")
print("Fichier envoyé avec succès !")

Fichier envoyé avec succès !


In [15]:
import pandas as pd
from io import BytesIO

# Télécharger le CSV des lignes en mémoire
obj = s3.get_object(Bucket="anfa-raw", Key="referentiel/lignes.csv")
df_lignes = pd.read_csv(BytesIO(obj["Body"].read()))
df_lignes

,ligne_id,nom,terminus_depart,terminus_arrivee,nb_arrets,distance_km
0,L01,Adidogomé - Adawlato,Adidogomé Assiyéyé,Adawlato Marché,9,9.89
1,L02,Agoè Assiyéyé - Grand Marché,Agoè Assiyéyé Terminus,Adawlato Grand Marché,9,16.28
2,L03,Avédji - Adawlato,Avédji Limousine,Adawlato Marché,9,11.67
3,L04,Université de Lomé - Adawlato,Campus Universitaire,Adawlato Marché,8,5.65
4,L05,Hédzranawoé - Adawlato,Hédzranawoé Terminus,Adawlato Marché,8,7.37
5,L06,Bè - Tokoin Casablanca,Bè Kpota,Tokoin Casablanca,7,5.49
6,L07,Cacavéli - BIA Centre,Cacavéli Marché,BIA Centre,7,10.03
7,L08,Aéroport GTA - Adawlato,Aéroport GTA,Adawlato Marché,7,8.73
8,L09,Baguida - Adawlato,Baguida Terminus,Adawlato Marché,8,13.13
9,L10,Anfamé - Université de Lomé,Anfamé Terminus,Campus Universitaire,8,5.89


In [16]:
# Top 3 des lignes les plus longues
df_lignes.nlargest(3, "distance_km")[["nom", "nb_arrets", "distance_km"]]

,nom,nb_arrets,distance_km
1,Agoè Assiyéyé - Grand Marché,9,16.28
8,Baguida - Adawlato,8,13.13
2,Avédji - Adawlato,9,11.67
